# Mean-Variance Portfolio Continuous-State Benchmark

Mean-variance reward benchmark with closed-form objectives and gradients. This is the high-noise continuous case: the paper role is to show the MSE/budget trade-off rather than hide it.

## Article Figure Set

This notebook is organized around the six exhibits that belong in the paper:

1. Main learning comparison: validation objective over training for simplex, logit MFREINFORCE, and REINFORCE.
2. Final performance table: objective, gap, policy/flow error when available, runtime, and budget.
3. Learned mean-field flow: population law over time against the optimal or target law.
4. Perturbation scale diagnostics: policy error, $J$ vs $J^\lambda$, and perturbation coverage.
5. Gradient decomposition: bias, variance, MSE, and the missing mean-field term exposed by the REINFORCE ablation.
6. Horizon and flow robustness: performance/runtime as $T$ and the nominal-flow estimator change.

Generalization checks are intentionally left for appendix-style follow-up cells; they are useful, but not central to the paper's claim.


In [ ]:

import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import pandas as pd
import torch

# Continuous benchmarks are small scalar recursions; CPU avoids kernel-launch noise.
torch.set_default_device("cpu")
torch.set_default_dtype(torch.float64)

from configs.portfolio import MAIN, MID, SMOKE
from mfc.environments.portfolio import Portfolio
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import apply_style, color_for, set_style, style_legend
from scripts.train import run_continuous
from scripts.test import (
    continuous_gradient_diagnostics,
    continuous_mean_field_term,
    continuous_objective_gap,
    continuous_perturbation_coverage,
    continuous_state_marginal_stability,
    group_by,
    load_runs,
)

set_style()

ENV_NAME = "portfolio"
ENV_CLASS = Portfolio
REF_LAM = 0.1
DIAGNOSTIC_REPS = 25
OBJECTIVE_SAMPLES = 2000
COVERAGE_SAMPLES = 5000


## Configuration and Data

Load the richest cached tier available: `main`, then `mid`, then `smoke`. Smoke runs are trained inline only when no cache exists.

In [ ]:

def cached_count(tier_name):
    run_dir = ROOT / "runs" / ENV_NAME / tier_name
    return len(list(run_dir.glob("*_seed*.pt")))

for tier_name, candidate in (("main", MAIN), ("mid", MID), ("smoke", SMOKE)):
    if cached_count(tier_name):
        tier, cfg = tier_name, candidate
        break
else:
    tier, cfg = "smoke", SMOKE

runs_dir = ROOT / "runs" / ENV_NAME / tier
for alg in cfg.algorithms:
    if tier == "smoke" and not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_continuous(ENV_NAME, alg, "smoke", progress="off")

runs = []
for alg in cfg.algorithms:
    runs.extend(load_runs(ENV_NAME, alg, tier, device="cpu"))

if not runs:
    raise RuntimeError(f"No runs found for {ENV_NAME}/{tier}; run scripts/train.py first.")

run_dtype = runs[0]["theta_final"].dtype
torch.set_default_dtype(run_dtype)
env = ENV_CLASS(dtype=run_dtype, device="cpu")

simplex_runs = [r for r in runs if r["alg"] == "simplex" and r["lam"] is not None]
ref_T = sorted({r["T"] for r in simplex_runs or runs})[len(sorted({r["T"] for r in simplex_runs or runs})) // 2]
available_lambdas = sorted({r["lam"] for r in simplex_runs if r["T"] == ref_T}) or sorted({r["lam"] for r in simplex_runs})
ref_lam = min(available_lambdas, key=lambda x: abs(float(x) - float(REF_LAM))) if available_lambdas else None
comparison_runs = [r for r in runs if r["T"] == ref_T]
reference_run = sorted([r for r in comparison_runs if r["alg"] == "simplex" and r["lam"] == ref_lam] or comparison_runs, key=lambda r: (r["seed"], r["alg"]))[0]

def optimal_theta_for(T):
    return env.riccati_optimal(T) if hasattr(env, "riccati_optimal") else env.optimal_theta(T)

theta_star = optimal_theta_for(ref_T)
optimal_J = env.exact_objective(theta_star, 0.0).item()
zero_J = env.exact_objective(torch.zeros(ref_T, 2, dtype=env.dtype), 0.0).item()
maximize = getattr(env, "MAXIMIZE", True)

print(f"tier: {tier}")
print(f"loaded runs: {len(runs)}")
print(f"article reference group: T={ref_T}, lambda={ref_lam}")
print(f"total cached training time: {sum(r['elapsed_seconds'] for r in runs):.1f}s")
print(f"optimal J^0(theta*): {optimal_J:.6g}; J^0(0): {zero_J:.6g}; objective is {'maximized' if maximize else 'minimized'}")


## 1. Main Learning Comparison

Validation objective over training for simplex across λ and the REINFORCE ablation. The closed-form optimum is shown as the reference line.

In [ ]:

fig, ax = viz.plot_validation_curve(comparison_runs, optimal_J=optimal_J)
ax.axhline(zero_J, color="0.25", linestyle=":", linewidth=1.3, label="zero policy")
style_legend(ax)
ax.set_title(f"{ENV_NAME}: validation objective (T={ref_T})")
fig


## 2. Final Performance Table

Final objective, gap to the closed-form optimum, captured improvement over the zero policy, parameter error, runtime, and equal-budget transition count.

In [ ]:

rows = []
for (alg, lam), group in sorted(group_by(comparison_runs, "alg", "lam").items(), key=lambda kv: (kv[0][0], -1 if kv[0][1] is None else kv[0][1])):
    final_J = torch.tensor([r["validation_J"][-1].item() for r in group], dtype=torch.float64)
    gap = (optimal_J - final_J.mean().item()) if maximize else (final_J.mean().item() - optimal_J)
    captured = ((final_J.mean().item() - zero_J) / (optimal_J - zero_J)) if maximize else ((zero_J - final_J.mean().item()) / (zero_J - optimal_J))
    theta_norm = torch.stack([(r["theta_final"] - theta_star).flatten().norm() for r in group]).mean().item()
    rows.append({
        "algorithm": alg,
        "lambda": "-" if lam is None else lam,
        "seeds": len(group),
        "final J mean": final_J.mean().item(),
        "final J std": final_J.std(unbiased=False).item() if len(group) > 1 else 0.0,
        "gap to optimal": gap,
        "captured improvement": captured,
        "mean ||theta-theta*||": theta_norm,
        "runtime mean s": sum(r["elapsed_seconds"] for r in group) / len(group),
        "transitions/step": cfg.transitions_per_step(ref_T) if alg == "simplex" else cfg.reinforce_B_equal_budget() * ref_T,
    })

performance_table = pd.DataFrame(rows)
performance_table["_lambda_sort"] = performance_table["lambda"].map(lambda x: -1.0 if x == "-" else float(x))
performance_table = performance_table.sort_values(["algorithm", "_lambda_sort"]).drop(columns="_lambda_sort")
performance_table


## 3. Learned Mean-Field Flow

The state law is summarized by its propagated mean and variance. This panel compares the learned flow to the closed-form optimal flow.

In [ ]:

theta_simplex = reference_run["theta_final"]
reinforce_candidates = [r for r in comparison_runs if r["alg"] == "reinforce"]
theta_reinforce = sorted(reinforce_candidates, key=lambda r: r["seed"])[0]["theta_final"] if reinforce_candidates else None

mu_star, Sigma_star = env.forward_moments(theta_star, 0.0)
mu_simplex, Sigma_simplex = env.forward_moments(theta_simplex, 0.0)

fig, axes = plt.subplots(1, 2 if theta_reinforce is not None else 1, figsize=(10, 3.8), constrained_layout=True)
axes = axes if hasattr(axes, "__len__") else [axes]
viz.plot_gaussian_flow(mu_simplex, Sigma_simplex, optimal_mu=mu_star, optimal_Sigma=Sigma_star, label="simplex", ax=axes[0])
axes[0].set_title(f"simplex, lambda={reference_run['lam']}")
if theta_reinforce is not None:
    mu_reinforce, Sigma_reinforce = env.forward_moments(theta_reinforce, 0.0)
    viz.plot_gaussian_flow(mu_reinforce, Sigma_reinforce, optimal_mu=mu_star, optimal_Sigma=Sigma_star, label="reinforce", ax=axes[1])
    axes[1].set_title("REINFORCE")
fig


## 4. Perturbation Scale Diagnostics

Closed-form $J^\lambda$ vs $J^0$ and the continuous transport-coverage analogue of the discrete $d_{TV}\leq\lambda$ check.

In [ ]:

lambda_runs = {}
for lam in sorted({r["lam"] for r in comparison_runs if r["alg"] == "simplex" and r["lam"] is not None}):
    candidates = [r for r in comparison_runs if r["alg"] == "simplex" and r["lam"] == lam]
    if candidates:
        lambda_runs[lam] = sorted(candidates, key=lambda r: r["seed"])[0]

if not lambda_runs:
    print("No simplex lambda sweep found for this horizon.")
else:
    lambdas = sorted(lambda_runs)
    learned_J0 = [env.exact_objective(lambda_runs[lam]["theta_final"], 0.0).item() for lam in lambdas]
    learned_Jlam = [env.exact_objective(lambda_runs[lam]["theta_final"], lam).item() for lam in lambdas]
    optimal_Jlam = [env.exact_objective(theta_star, lam).item() for lam in lambdas]

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
    axes[0].plot(lambdas, learned_J0, marker="o", label=r"$J^0(\hat\theta_\lambda)$")
    axes[0].plot(lambdas, learned_Jlam, marker="s", label=r"$J^\lambda(\hat\theta_\lambda)$")
    axes[0].plot(lambdas, optimal_Jlam, marker="^", label=r"$J^\lambda(\theta^*)$")
    axes[0].axhline(optimal_J, color="0.2", linestyle="--", label=r"$J^0(\theta^*)$")
    apply_style(axes[0], xlabel="perturbation scale lambda", ylabel="objective")
    style_legend(axes[0])
    axes[0].set_title(r"$J$ vs $J^\lambda$")

    coverage = continuous_perturbation_coverage(env, theta_star, lam=ref_lam, n_samples=COVERAGE_SAMPLES,
                                                generator=torch.Generator(device="cpu").manual_seed(0))
    times = [row["t"] for row in coverage]
    axes[1].plot(times, [row["mean_W2"].item() for row in coverage], marker="o", label="mean W2")
    axes[1].plot(times, [row["max_W2"].item() for row in coverage], marker="s", label="max W2")
    apply_style(axes[1], xlabel="time step", ylabel="transport perturbation size")
    style_legend(axes[1])
    axes[1].set_title(f"coverage check at lambda={ref_lam}")
    fig

    display(pd.DataFrame([{k: (v.item() if torch.is_tensor(v) and v.ndim == 0 else v) for k, v in row.items()} for row in coverage]).head())


## 5. Gradient Decomposition and Missing Mean-Field Term

Closed-form gradients let this notebook split simplex bias into perturbation and estimation pieces, then measure the term REINFORCE drops by a paired comparison on the same rollouts.

In [ ]:

theta_probe = theta_star
simplex_diag = continuous_gradient_diagnostics(
    env, theta_probe, lam=ref_lam, B=cfg.B, n_aux=cfg.n_aux, reps=DIAGNOSTIC_REPS,
    algorithm="simplex", baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(123),
)
reinforce_diag = continuous_gradient_diagnostics(
    env, theta_probe, lam=0.0, B=cfg.reinforce_B_equal_budget(), reps=DIAGNOSTIC_REPS,
    algorithm="reinforce", baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(456),
)
omitted = continuous_mean_field_term(
    env, theta_probe, lam=ref_lam, B=cfg.B, reps=DIAGNOSTIC_REPS,
    baseline=cfg.baseline, generator=torch.Generator(device="cpu").manual_seed(789),
)

gradient_table = pd.DataFrame([
    {"quantity": "simplex bias", "norm": simplex_diag["bias"].norm().item(), "std/se norm": simplex_diag["bias_se"].norm().item(), "mse": simplex_diag["mse"].mean().item()},
    {"quantity": "simplex perturbation bias", "norm": simplex_diag["perturbation_bias"].norm().item(), "std/se norm": 0.0, "mse": float("nan")},
    {"quantity": "simplex estimation bias", "norm": simplex_diag["estimation_bias"].norm().item(), "std/se norm": simplex_diag["bias_se"].norm().item(), "mse": float("nan")},
    {"quantity": "reinforce bias", "norm": reinforce_diag["bias"].norm().item(), "std/se norm": reinforce_diag["bias_se"].norm().item(), "mse": reinforce_diag["mse"].mean().item()},
    {"quantity": "omitted mean-field term", "norm": omitted["mean"].norm().item(), "std/se norm": omitted["se"].norm().item(), "mse": float("nan")},
])
gradient_table


## 6. Horizon Robustness

Continuous benchmarks have no exact-vs-particle axis; the robustness exhibit is horizon scaling under the equal-budget allocation.

In [ ]:

rows = []
for (alg, T, lam), group in sorted(group_by(runs, "alg", "T", "lam").items(), key=lambda kv: (kv[0][1], kv[0][0], -1 if kv[0][2] is None else kv[0][2])):
    theta_star_T = optimal_theta_for(T)
    optimal_J_T = env.exact_objective(theta_star_T, 0.0).item()
    final_J = torch.tensor([r["validation_J"][-1].item() for r in group], dtype=torch.float64)
    gap = (optimal_J_T - final_J.mean().item()) if maximize else (final_J.mean().item() - optimal_J_T)
    rows.append({
        "algorithm": alg,
        "T": T,
        "lambda": "-" if lam is None else lam,
        "seeds": len(group),
        "final J mean": final_J.mean().item(),
        "gap to optimal": gap,
        "runtime mean s": sum(r["elapsed_seconds"] for r in group) / len(group),
    })

scaling_table = pd.DataFrame(rows)
display(scaling_table)

fig, ax = plt.subplots(figsize=(6.8, 4.0), constrained_layout=True)
for i, ((alg, lam), sub) in enumerate(scaling_table.groupby(["algorithm", "lambda"], dropna=False)):
    if len(sub["T"].unique()) < 2:
        continue
    sub = sub.sort_values("T")
    ax.plot(sub["T"], sub["gap to optimal"], marker="o", color=color_for(i), label=f"{alg}, lambda={lam}")
apply_style(ax, xlabel="horizon T", ylabel="gap to optimal")
style_legend(ax)
ax.set_title("Horizon scaling across cached runs")
fig


## Runtime

A small footer keeps notebook runtime reportable in the article artifact.

In [ ]:

print(f"Notebook wall time: {time.perf_counter() - _notebook_start:.1f}s")
print("Paper figures covered: validation, final table, population flow, lambda diagnostics, gradient decomposition, horizon/flow robustness.")
